In [ ]:
%load_ext sql
%sql duckdb:///:memory:

In [ ]:
%%sql
CREATE VIEW decile_ref AS SELECT * FROM read_csv_auto('../reports/tableau/decile_lift_reference.csv');

In [ ]:
%%sql
CREATE VIEW submission AS SELECT * FROM read_csv_auto('../data/output/submission_v5_rank.csv');


In [ ]:
%%sql
CREATE VIEW test AS SELECT * FROM read_csv_auto('../data/input/test.csv');

In [ ]:
%%sql
SELECT count(*) FROM oof;

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT SK_ID_CURR) AS unique_ids,
    SUM(CASE WHEN blend_score IS NULL THEN 1 ELSE 0 END) AS null_scores,
    SUM(CASE WHEN TARGET IS NULL THEN 1 ELSE 0 END) AS null_targets,
    ROUND(AVG(TARGET)* 100,2) AS default_rate_pct
FROM oof;

In [ ]:
%%sql
WITH deciles AS (
    SELECT 
        TARGET,
        blend_score,
        NTILE(10) OVER (ORDER BY blend_score DESC) AS decile
    FROM oof
)
SELECT 
    decile,
    COUNT(*) AS n,
    SUM(TARGET) AS defaults,
    ROUND(AVG(blend_score), 4) AS avg_score,
    ROUND(AVG(TARGET), 4) AS default_rate,
    -- Calculamos el lift dividiendo la tasa del decil entre la tasa global (~0.0807)
    ROUND(AVG(TARGET) / (SELECT AVG(TARGET) FROM oof), 2) AS lift
FROM deciles
GROUP BY decile
ORDER BY decile;

In [ ]:
%%sql
WITH deciles AS (
    SELECT 
        TARGET,
        NTILE(10) OVER (ORDER BY blend_score DESC) AS decile
    FROM oof
),
agg AS (
    SELECT 
        decile,
        COUNT(*) AS n,
        SUM(TARGET) AS defaults,
        COUNT(*) - SUM(TARGET) AS non_defaults
    FROM deciles
    GROUP BY decile
),
totals AS (
    SELECT 
        SUM(n) AS total_n,
        SUM(defaults) AS total_defaults,
        SUM(non_defaults) AS total_non_defaults
    FROM agg
)
SELECT 
    a.decile,
    a.n,
    a.defaults,
    -- Porcentaje acumulado de poblacion
    ROUND(SUM(a.n) OVER (ORDER BY a.decile) * 100.0 / t.total_n, 2) AS cum_pop_pct,
    -- Porcentaje acumulado de malos (impagos)
    ROUND(SUM(a.defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_defaults, 2) AS cum_defaults_pct,
    -- Porcentaje acumulado de buenos (no impagos)
    ROUND(SUM(a.non_defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_non_defaults, 2) AS cum_non_defaults_pct,
    -- Diferencia para calcular el KS
    ROUND(
        (SUM(a.defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_defaults) - 
        (SUM(a.non_defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_non_defaults), 
    2) AS ks_diff
FROM agg a
CROSS JOIN totals t
ORDER BY a.decile;

Task 4 Real Data vs Prediction

In [ ]:
%%sql
WITH base AS (
    SELECT 
        o.TARGET,
        o.p_catboost,
        t.NAME_INCOME_TYPE,
        t.NAME_EDUCATION_TYPE,
        -- Tramos de edad (DAYS_BIRTH es negativo)
        CASE 
            WHEN t.DAYS_BIRTH > -10950 THEN 'Under 30'
            WHEN t.DAYS_BIRTH > -14600 THEN '30-40'
            WHEN t.DAYS_BIRTH > -18250 THEN '40-50'
            ELSE '50+'
        END AS age_group,
        -- Tramos de EXT_SOURCE_2
        CASE 
            WHEN t.EXT_SOURCE_2 < 0.3 THEN 'Low (<0.3)'
            WHEN t.EXT_SOURCE_2 < 0.6 THEN 'Medium (0.3-0.6)'
            WHEN t.EXT_SOURCE_2 >= 0.6 THEN 'High (>=0.6)'
            ELSE 'Missing'
        END AS ext2_group
    FROM oof o
    JOIN train t ON o.SK_ID_CURR = t.SK_ID_CURR
),
segments AS (
    -- Unimos los distintos tipos de segmentos para tenerlos en una sola tabla
    SELECT 'Income Type' AS segment_type, NAME_INCOME_TYPE AS segment_value, TARGET, p_catboost FROM base
    UNION ALL
    SELECT 'Education' AS segment_type, NAME_EDUCATION_TYPE AS segment_value, TARGET, p_catboost FROM base
    UNION ALL
    SELECT 'Age Group' AS segment_type, age_group AS segment_value, TARGET, p_catboost FROM base
    UNION ALL
    SELECT 'EXT_SOURCE_2' AS segment_type, ext2_group AS segment_value, TARGET, p_catboost FROM base
)
SELECT 
    segment_type,
    segment_value,
    COUNT(*) AS n,
    ROUND(AVG(TARGET) * 100, 2) AS real_default_rate_pct,
    ROUND(AVG(p_catboost) * 100, 2) AS predicted_default_rate_pct
FROM segments
GROUP BY segment_type, segment_value
-- Regla de la guía: mínimo 100 personas por segmento
HAVING COUNT(*) >= 100
ORDER BY segment_type, real_default_rate_pct DESC;

Segment Risk in the Test(Submission)

In [ ]:
%%sql
WITH base AS (
    SELECT 
        s.TARGET AS blend_score,
        t.NAME_INCOME_TYPE,
        t.NAME_EDUCATION_TYPE,
        CASE 
            WHEN t.DAYS_BIRTH > -10950 THEN 'Under 30'
            WHEN t.DAYS_BIRTH > -14600 THEN '30-40'
            WHEN t.DAYS_BIRTH > -18250 THEN '40-50'
            ELSE '50+'
        END AS age_group,
        PERCENT_RANK() OVER (ORDER BY s.TARGET DESC) AS risk_percent_rank
    FROM submission s
    JOIN test t ON s.SK_ID_CURR = t.SK_ID_CURR
),
segments AS (
    -- Aquí creamos la columna segment_type que faltaba
    SELECT 'Income Type' AS segment_type, NAME_INCOME_TYPE AS segment_value, blend_score, risk_percent_rank FROM base
    UNION ALL
    SELECT 'Education' AS segment_type, NAME_EDUCATION_TYPE AS segment_value, blend_score, risk_percent_rank FROM base
    UNION ALL
    SELECT 'Age Group' AS segment_type, age_group AS segment_value, blend_score, risk_percent_rank FROM base
)
SELECT 
    segment_type,
    segment_value,
    COUNT(*) AS n,
    ROUND(AVG(blend_score), 4) AS avg_predicted_score,
    ROUND(AVG(CASE WHEN risk_percent_rank <= 0.10 THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_in_top_10_risk
FROM segments
GROUP BY segment_type, segment_value
HAVING COUNT(*) >= 100
ORDER BY segment_type, avg_predicted_score DESC;

Export Aggregated Results to CSV    


In [ ]:
%%sql
COPY (
    WITH deciles AS (
        SELECT 
            TARGET, 
            blend_score,
            NTILE(10) OVER (ORDER BY blend_score DESC) AS decile
        FROM oof
    ),
    agg AS (
        SELECT 
            decile, 
            COUNT(*) AS n, 
            SUM(TARGET) AS defaults, 
            COUNT(*) - SUM(TARGET) AS non_defaults,
            AVG(blend_score) AS avg_score,
            AVG(TARGET) AS default_rate
        FROM deciles
        GROUP BY decile
    ),
    totals AS (
        SELECT SUM(n) AS total_n, SUM(defaults) AS total_defs, SUM(non_defaults) AS total_non_defs FROM agg
    )
    SELECT 
        a.decile, 
        a.n, 
        a.defaults, 
        ROUND(a.avg_score, 4) AS avg_score,
        ROUND(a.default_rate, 4) AS default_rate,
        ROUND(a.default_rate / (SELECT AVG(TARGET) FROM oof), 2) AS lift,
        ROUND(SUM(a.n) OVER (ORDER BY a.decile) * 100.0 / t.total_n, 2) AS cum_pop_pct,
        ROUND(SUM(a.defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_defs, 2) AS cum_defaults_pct,
        ROUND(SUM(a.non_defaults) OVER (ORDER BY a.decile) * 100.0 / t.total_non_defs, 2) AS cum_non_defaults_pct
    FROM agg a
    CROSS JOIN totals t
    ORDER BY a.decile
) TO '../reports/tableau/decile_lift.csv' (HEADER, DELIMITER ',');

In [ ]:
%%sql
COPY (
    WITH base AS (
        SELECT 
            o.TARGET,
            o.p_catboost,
            t.NAME_INCOME_TYPE,
            t.NAME_EDUCATION_TYPE,
            CASE 
                WHEN t.DAYS_BIRTH > -10950 THEN 'Under 30'
                WHEN t.DAYS_BIRTH > -14600 THEN '30-40'
                WHEN t.DAYS_BIRTH > -18250 THEN '40-50'
                ELSE '50+'
            END AS age_group,
            CASE 
                WHEN t.EXT_SOURCE_2 < 0.3 THEN 'Low (<0.3)'
                WHEN t.EXT_SOURCE_2 < 0.6 THEN 'Medium (0.3-0.6)'
                WHEN t.EXT_SOURCE_2 >= 0.6 THEN 'High (>=0.6)'
                ELSE 'Missing'
            END AS ext2_group
        FROM oof o
        JOIN train t ON o.SK_ID_CURR = t.SK_ID_CURR
    ),
    segments AS (
        SELECT 'Income Type' AS segment_type, NAME_INCOME_TYPE AS segment_value, TARGET, p_catboost FROM base
        UNION ALL
        SELECT 'Education' AS segment_type, NAME_EDUCATION_TYPE AS segment_value, TARGET, p_catboost FROM base
        UNION ALL
        SELECT 'Age Group' AS segment_type, age_group AS segment_value, TARGET, p_catboost FROM base
        UNION ALL
        SELECT 'EXT_SOURCE_2' AS segment_type, ext2_group AS segment_value, TARGET, p_catboost FROM base
    )
    SELECT 
        segment_type,
        segment_value,
        COUNT(*) AS n,
        ROUND(AVG(TARGET) * 100, 2) AS real_default_rate_pct,
        ROUND(AVG(p_catboost) * 100, 2) AS predicted_default_rate_pct
    FROM segments
    GROUP BY segment_type, segment_value
    HAVING COUNT(*) >= 100
    ORDER BY segment_type, real_default_rate_pct DESC
) TO '../reports/tableau/segment_risk_train.csv' (HEADER, DELIMITER ',');

In [ ]:
%%sql
COPY (
    WITH base AS (
        SELECT 
            s.TARGET AS blend_score,
            t.NAME_INCOME_TYPE,
            t.NAME_EDUCATION_TYPE,
            CASE 
                WHEN t.DAYS_BIRTH > -10950 THEN 'Under 30'
                WHEN t.DAYS_BIRTH > -14600 THEN '30-40'
                WHEN t.DAYS_BIRTH > -18250 THEN '40-50'
                ELSE '50+'
            END AS age_group,
            PERCENT_RANK() OVER (ORDER BY s.TARGET DESC) AS risk_percent_rank
        FROM submission s
        JOIN test t ON s.SK_ID_CURR = t.SK_ID_CURR
    ),
    segments AS (
        SELECT 'Income Type' AS segment_type, NAME_INCOME_TYPE AS segment_value, blend_score, risk_percent_rank FROM base
        UNION ALL
        SELECT 'Education' AS segment_type, NAME_EDUCATION_TYPE AS segment_value, blend_score, risk_percent_rank FROM base
        UNION ALL
        SELECT 'Age Group' AS segment_type, age_group AS segment_value, blend_score, risk_percent_rank FROM base
    )
    SELECT 
        segment_type,
        segment_value,
        COUNT(*) AS n,
        ROUND(AVG(blend_score), 4) AS avg_predicted_score,
        ROUND(AVG(CASE WHEN risk_percent_rank <= 0.10 THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_in_top_10_risk
    FROM segments
    GROUP BY segment_type, segment_value
    HAVING COUNT(*) >= 100
    ORDER BY segment_type, avg_predicted_score DESC
) TO '../reports/tableau/segment_risk_test.csv' (HEADER, DELIMITER ',');